In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score
import string
import re

In [2]:
df = pd.read_csv('train.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
columns_to_keep = {'PassengerId', 'Name','Survived', 'Sex', 'Ticket', 'Fare', 'Cabin', 'Embarked'}
renamed_columns = {'Pclass': 'PassengerClass', 'Sibsp': 'SiblingsSpousesAboard', 'Parch': 'ParentsChildrenAboard'}

In [4]:
renamed_columns = {'Pclass': 'PassengerClass', 'Sibsp': 'SiblingsSpousesAboard', 'Parch': 'ParentsChildrenAboard'}
df.rename(columns=renamed_columns, inplace=True)

In [5]:
for col in df.columns:
    if df[col].dtype == 'object': #only check text columns
        weird = df[col].dropna().apply(lambda x: bool(re.search(r'[^a-zA-Z0-9\s\.,\-]', x))) #check for non-ASCII characters
        if weird.any():
            print(f"Column '{col}' has weird characters in these entries:")
            

Column 'Name' has weird characters in these entries:
Column 'Ticket' has weird characters in these entries:


In [6]:
df = df.drop(columns=['Name', 'Ticket', 'Cabin', 'PassengerId'])

In [7]:
df.duplicated().sum() #checking for duplicates

np.int64(111)

In [8]:
df = df.drop_duplicates() #removing duplicates
df.duplicated().sum() #checking again for duplicates

np.int64(0)

In [9]:
df.isnull().sum() #finding if there is NAN values

Survived                   0
PassengerClass             0
Sex                        0
Age                      104
SibSp                      0
ParentsChildrenAboard      0
Fare                       0
Embarked                   2
dtype: int64

In [10]:
df['Age'] =df['Age'].fillna(df['Age'].median())
df['Embarked'] =df['Embarked'].fillna(df['Embarked'].mode()[0])

In [11]:
df.isnull().sum() #finding any remaining nulls

Survived                 0
PassengerClass           0
Sex                      0
Age                      0
SibSp                    0
ParentsChildrenAboard    0
Fare                     0
Embarked                 0
dtype: int64

In [12]:
df.head()

,Survived,PassengerClass,Sex,Age,SibSp,ParentsChildrenAboard,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [13]:
df.dtypes #understanding the data types and whether we need to convert any for modeling

Survived                   int64
PassengerClass             int64
Sex                       object
Age                      float64
SibSp                      int64
ParentsChildrenAboard      int64
Fare                     float64
Embarked                  object
dtype: object

In [14]:
df['Sex'] = df['Sex'].map({'male' : 1, 'female' : 0})
df['Embarked'] = df['Embarked'].map({'S' : 0, 'C' : 1, 'Q' : 2})
df['Embarked'] = df['Embarked'].fillna(0)
df['Embarked'] = df['Embarked'].astype(int)

In [15]:
df.head() #seeing whether it converted correctly

,Survived,PassengerClass,Sex,Age,SibSp,ParentsChildrenAboard,Fare,Embarked
0,0,3,1,22.0,1,0,7.2500,0
1,1,1,0,38.0,1,0,71.2833,1
2,1,3,0,26.0,0,0,7.9250,0
3,1,1,0,35.0,1,0,53.1000,0
4,0,3,1,35.0,0,0,8.0500,0


In [16]:
df.info() #double check the data types and look for any remaining nulls

<class 'pandas.core.frame.DataFrame'>
Index: 780 entries, 0 to 890
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Survived               780 non-null    int64  
 1   PassengerClass         780 non-null    int64  
 2   Sex                    780 non-null    int64  
 3   Age                    780 non-null    float64
 4   SibSp                  780 non-null    int64  
 5   ParentsChildrenAboard  780 non-null    int64  
 6   Fare                   780 non-null    float64
 7   Embarked               780 non-null    int64  
dtypes: float64(2), int64(6)
memory usage: 54.8 KB


In [17]:
df.describe() #checking for outliers

,Survived,PassengerClass,Sex,Age,SibSp,ParentsChildrenAboard,Fare,Embarked
count,780.000000,780.000000,780.000000,780.000000,780.000000,780.000000,780.000000,780.000000
mean,0.412821,2.246154,0.625641,29.571051,0.525641,0.417949,34.829108,0.347436
std,0.492657,0.854452,0.484267,13.722689,0.988046,0.838536,52.263440,0.613126
min,0.000000,1.000000,0.000000,0.420000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,1.000000,0.000000,21.750000,0.000000,0.000000,8.050000,0.000000
50%,0.000000,3.000000,1.000000,28.000000,0.000000,0.000000,15.950000,0.000000
75%,1.000000,3.000000,1.000000,36.000000,1.000000,1.000000,34.375000,1.000000
max,1.000000,3.000000,1.000000,80.000000,8.000000,6.000000,512.329200,2.000000


In [18]:
#outlier detections
mean_fare = df['Fare'].mean()
std_fare = df['Fare'].std()
cutoff = std_fare * 3
df['Fare_outlier'] = df['Fare'].apply(lambda x: abs(x - mean_fare) > cutoff)
outliers = df[df['Fare_outlier']]

In [19]:
print(outliers[['Fare', 'Fare_outlier']])

         Fare  Fare_outlier
27   263.0000          True
88   263.0000          True
118  247.5208          True
258  512.3292          True
299  247.5208          True
311  262.3750          True
341  263.0000          True
377  211.5000          True
380  227.5250          True
438  263.0000          True
527  221.7792          True
557  227.5250          True
679  512.3292          True
689  211.3375          True
700  227.5250          True
716  227.5250          True
730  211.3375          True
737  512.3292          True
742  262.3750          True
779  211.3375          True


In [20]:
#cap the outliers
fare_cap = df['Fare'].quantile(0.99) #cap at 99th percentile
df['Fare'] = df['Fare'].clip(upper=fare_cap)

#drop the outlier flag column
df.drop(columns='Fare_outlier', inplace=True)

In [21]:
df.describe() #see if the cap works or not

,Survived,PassengerClass,Sex,Age,SibSp,ParentsChildrenAboard,Fare,Embarked
count,780.000000,780.000000,780.000000,780.000000,780.000000,780.000000,780.000000,780.000000
mean,0.412821,2.246154,0.625641,29.571051,0.525641,0.417949,33.864540,0.347436
std,0.492657,0.854452,0.484267,13.722689,0.988046,0.838536,45.281324,0.613126
min,0.000000,1.000000,0.000000,0.420000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,1.000000,0.000000,21.750000,0.000000,0.000000,8.050000,0.000000
50%,0.000000,3.000000,1.000000,28.000000,0.000000,0.000000,15.950000,0.000000
75%,1.000000,3.000000,1.000000,36.000000,1.000000,1.000000,34.375000,1.000000
max,1.000000,3.000000,1.000000,80.000000,8.000000,6.000000,262.375000,2.000000


In [22]:
df = df.drop_duplicates() #removing duplicates
df.duplicated().sum() #checking again for duplicates

np.int64(0)

In [23]:
print("Shape:", df.shape)
print("\nMissing values:\n", df.isnull().sum())
print("\nDuplicates:", df.duplicated().sum())
print("\nDtypes:\n", df.dtypes)
df.describe()

Shape: (775, 8)

Missing values:
 Survived                 0
PassengerClass           0
Sex                      0
Age                      0
SibSp                    0
ParentsChildrenAboard    0
Fare                     0
Embarked                 0
dtype: int64

Duplicates: 0

Dtypes:
 Survived                   int64
PassengerClass             int64
Sex                        int64
Age                      float64
SibSp                      int64
ParentsChildrenAboard      int64
Fare                     float64
Embarked                   int64
dtype: object


,Survived,PassengerClass,Sex,Age,SibSp,ParentsChildrenAboard,Fare,Embarked
count,775.000000,775.000000,775.000000,775.000000,775.000000,775.000000,775.000000,775.000000
mean,0.412903,2.246452,0.623226,29.581187,0.529032,0.420645,33.907613,0.349677
std,0.492674,0.853574,0.484890,13.766359,0.990326,0.840565,45.401205,0.614465
min,0.000000,1.000000,0.000000,0.420000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,1.000000,0.000000,21.000000,0.000000,0.000000,8.050000,0.000000
50%,0.000000,3.000000,1.000000,28.000000,0.000000,0.000000,15.900000,0.000000
75%,1.000000,3.000000,1.000000,36.000000,1.000000,1.000000,34.197900,1.000000
max,1.000000,3.000000,1.000000,80.000000,8.000000,6.000000,262.375000,2.000000


In [24]:
df.to_csv('train_clean.csv', index=False)
print("Clean dataset saved as 'train_clean.csv'")

Clean dataset saved as 'train_clean.csv'
